# TP Jour 1 — Choix des Datasets Kaggle (Mono ou Multi-Sources), HDFS & Couche Bronze

Bienvenue dans ce fil rouge **Data Lakehouse** ! Vous êtes libres de choisir votre thématique métier et de combiner **un dataset multi-tables ou plusieurs datasets publics complémentaires sur Kaggle** afin de les croiser par jointure.

### Objectifs de la séance
1. **Cadrer votre sujet métier** et sélectionner **un ou plusieurs datasets Kaggle** (~50 à 500 Mo au total) partageant au moins un axe de jointure (date/heure, zone géographique, code ISO/catégorie, identifiant métier).
2. **Explorer le cluster HDFS réel** (`1 NameNode` + `3 DataNodes`) configuré avec `dfs.blocksize = 16 Mo` et `dfs.replication = 2`.
3. **Ingérer vos fichiers bruts multi-sources sur HDFS** (`/lakehouse/bronze/raw/<source>/...`) et observer physiquement leur découpage en blocs de 16 Mo.
4. **Réaliser un crash-test en direct** : couper `datanode2` et observer la détection de panne + l'auto-réparation vers le 3e DataNode en ~50 secondes.
5. **Construire votre couche Bronze** (`/lakehouse/bronze/`) au format **Delta Lake** avec traçabilité de la source (`_source_dataset`, `_source_file`, `_ingested_at`) et inspecter vos jobs sur **Spark Web UI** (`http://localhost:4040`).

---
### Fiche de cadrage de votre projet *(À compléter par le groupe)*
- **Groupe :** Walid Hdilou · Joud Atallah
- **Titre du sujet :** `Qu'est-ce qui fait le succès d'un jeu sur Steam ? Prix, genre, ancienneté et réception par les joueurs`
- **Dataset(s) Kaggle retenu(s) (`auteur/nom-du-dataset`) :**
  1. Source principale : `andrewmvd/steam-reviews` — avis joueurs (table de faits, `steam_reviews.csv`, ~105 Mo, ~3,76 M avis échantillonnés uniformément, seed 42, sur les jeux présents dans le catalogue)
  2. Source(s) complémentaire(s) pour jointure : `nikdavis/steam-store-games` — catalogue Steam (table de dimension, `steam_games.csv`, ~5,8 Mo, 27 075 jeux)
- **Clé(s) de jointure envisagée(s) entre sources / tables :** `steam_reviews.app_id = steam_games.appid` *(identifiant métier du jeu Steam ; jointure N-1 faits → dimension, candidate au `broadcast` car la dimension fait < 10 Mo)*
- **Problématique métier principale :** `Quels facteurs du catalogue (prix, genre, éditeur, plateformes, ancienneté de sortie) sont associés à une meilleure réception des joueurs (part d'avis positifs, votes « utiles »), et peut-on segmenter les jeux en profils distincts (blockbusters, niches appréciées, jeux décevants…) ?`
- **Variables clés retenues (temporelles, quantitatives, catégorielles) :**
  - *Temporelle :* `release_date` (catalogue) → année / ancienneté de sortie. ⚠️ La version Kaggle des avis ne contient **pas** d'horodatage d'avis : l'axe temps provient de la jointure avec le catalogue.
  - *Quantitatives :* `price`, `positive_ratings`, `negative_ratings`, `average_playtime`, `median_playtime`, `achievements`, `review_votes`
  - *Catégorielles :* `genres`, `categories`, `steamspy_tags`, `developer`, `publisher`, `platforms`, `owners` (tranche), `required_age`, `review_score` (1 = recommandé / -1 = non recommandé)

## 1. Téléchargement de vos Datasets Kaggle dans `/workspace/data_raw/<source>/`

Renseignez dans la liste `KAGGLE_DATASETS` un ou plusieurs identifiants Kaggle (ex. `["auteur1/dataset-principal", "auteur2/meteo-ou-referentiel"]`), ou déposez directement vos fichiers/sous-dossiers dans `data_raw/`.
- Chaque dataset est isolé dans son propre sous-dossier `data_raw/<nom-dataset>/` pour éviter toute collision de noms (`train.csv`, `metadata.csv`, etc.) entre plusieurs sources.
- **Critère important :** Vérifiez qu'au moins un fichier dépasse **20 à 32 Mo** afin d'observer sa division en plusieurs blocs HDFS de **16 Mo** !

In [1]:
# Garde-fou d'environnement : ce notebook doit tourner dans le conteneur spark-jupyter
# (kernel du serveur Jupyter http://localhost:8888), jamais sur un Python local Windows :
# celui-ci n'a ni PySpark ni accès à hdfs://namenode:9000, et créerait un dossier D:\workspace\ vide.
import sys
from pathlib import Path

IN_CONTAINER = sys.platform == "linux" and Path("/opt/spark/conf/spark-defaults.conf").is_file()
assert IN_CONTAINER, (
    f"Kernel local détecté ({sys.executable}). Dans VS Code : Select Kernel -> Existing Jupyter Server -> "
    "http://localhost:8888/?token=bigdata -> « Python 3.15 (PySpark 4.2 + Delta Lake 4.4) »."
)
print(f"Kernel conteneur OK : Python {sys.version.split()[0]} ({sys.executable})")


Kernel conteneur OK : Python 3.15.2 (/opt/venv/bin/python)


In [2]:
import re
import shutil
from pathlib import Path

DATA_RAW = Path("/workspace/data_raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)
SUPPORTED_EXTS = {".csv", ".parquet", ".json", ".jsonl", ".tsv"}

# Sources Kaggle du projet. Les fichiers sont déjà préparés localement dans
# data_raw/steam-store-games/ et data_raw/steam-reviews/ (sous-dossier = fin du slug) :
# la boucle ci-dessous les détecte et ne relance donc aucun téléchargement.
KAGGLE_DATASETS: list[str] = [
    "andrewmvd/steam-reviews",  # faits : avis joueurs (filtrés + échantillonnés ~100 Mo)
    "nikdavis/steam-store-games",  # dimension : catalogue des jeux (steam.csv -> steam_games.csv)
]

for slug in KAGGLE_DATASETS:
    slug = slug.strip()
    if not slug:
        continue
    subdir = re.sub(r"[^a-zA-Z0-9._-]+", "_", slug.split("/")[-1]).strip("_") or "dataset"
    target_dir = DATA_RAW / subdir
    existing = [f for f in target_dir.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS] if target_dir.exists() else []
    if not existing:
        import kagglehub
        print(f"Téléchargement de {slug} -> {target_dir}/ ...")
        cache_dir = Path(kagglehub.dataset_download(slug))
        target_dir.mkdir(parents=True, exist_ok=True)
        for src_file in sorted(cache_dir.rglob("*")):
            if src_file.is_file() and src_file.suffix.lower() in SUPPORTED_EXTS:
                dest_file = target_dir / src_file.relative_to(cache_dir)
                dest_file.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src_file, dest_file)

local_files = sorted([f for f in DATA_RAW.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS])

assert len(local_files) > 0, (
    "Aucun fichier détecté dans /workspace/data_raw/. "
    "Renseignez `KAGGLE_DATASETS` ci-dessus ou utilisez `task data:download DATASET='auteur1/ds1,auteur2/ds2'`."
)

print(f"Inventaire local multi-sources dans {DATA_RAW} ({len(local_files)} fichier(s)) :")
for f in local_files:
    rel = f.relative_to(DATA_RAW)
    size_mb = f.stat().st_size / (1024 * 1024)
    expected_blocks = max(1, int(-(-f.stat().st_size // (16 * 1024 * 1024))))
    print(f" - {str(rel):<55} {size_mb:>7.2f} Mo  (~{expected_blocks} bloc(s) HDFS de 16 Mo)")

Inventaire local multi-sources dans /workspace/data_raw (2 fichier(s)) :
 - steam-reviews/steam_reviews.csv                          100.35 Mo  (~7 bloc(s) HDFS de 16 Mo)
 - steam-store-games/steam_games.csv                          5.55 Mo  (~1 bloc(s) HDFS de 16 Mo)


## 2. Démarrage de la `SparkSession` pré-configurée sur HDFS

Grâce au fichier `config/spark-defaults.conf`, Spark pointe automatiquement vers `hdfs://namenode:9000` avec **Delta Lake** activé.

### Exercice 2.1 — Initialiser `SparkSession` et contrôler la configuration HDFS
1. Créez une session Spark nommée `"J1_HDFS_Ingestion_Bronze"`.
2. Récupérez via `spark.conf.get(...)` la valeur de `"spark.hadoop.fs.defaultFS"`, `"spark.hadoop.dfs.blocksize"` et `"spark.hadoop.dfs.replication"`.

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# La config HDFS + Delta est déjà portée par config/spark-defaults.conf : appName + getOrCreate suffisent.
spark = SparkSession.builder.appName("J1_HDFS_Ingestion_Bronze").getOrCreate()

assert spark is not None, "TODO : Initialisez `spark` avec SparkSession.builder..."

default_fs = spark.conf.get("spark.hadoop.fs.defaultFS")
block_size = spark.conf.get("spark.hadoop.dfs.blocksize")
replication = spark.conf.get("spark.hadoop.dfs.replication")

print("Spark version       :", spark.version)
print("Default FileSystem  :", default_fs)
print("HDFS Block Size (o) :", block_size)
print("HDFS Replication    :", replication)

assert default_fs == "hdfs://namenode:9000"
assert int(block_size) == 16 * 1024 * 1024, "Bloc HDFS attendu : 16 Mo"
assert int(replication) == 2


:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5abb6319-2504-4d8d-b722-3add1703ddf8;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found

Spark version       : 4.2.0
Default FileSystem  : hdfs://namenode:9000
HDFS Block Size (o) : 16777216
HDFS Replication    : 2


## 3. Copie des fichiers bruts vers HDFS & Inspection des Blocs de 16 Mo

La cellule suivante téléverse vos fichiers bruts de `/workspace/data_raw` vers `/lakehouse/bronze/raw/` sur HDFS, puis interroge le NameNode pour afficher la localisation exacte de chaque bloc de 16 Mo sur les 3 DataNodes (`datanode1`, `datanode2`, `datanode3`).

In [4]:
jvm = spark._jvm
hadoop_conf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
PathHDFS = jvm.org.apache.hadoop.fs.Path

raw_hdfs_dir = PathHDFS("/lakehouse/bronze/raw")
fs.mkdirs(raw_hdfs_dir)

for file_path in local_files:
    rel_posix = file_path.relative_to(DATA_RAW).as_posix()
    src = PathHDFS(f"file://{file_path}")
    dst = PathHDFS(f"/lakehouse/bronze/raw/{rel_posix}")
    fs.mkdirs(dst.getParent())
    fs.copyFromLocalFile(False, True, src, dst)
    print(f"Téléversé vers HDFS : {dst}")

Téléversé vers HDFS : /lakehouse/bronze/raw/steam-reviews/steam_reviews.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/steam-store-games/steam_games.csv


In [5]:
# Inspection récursive du découpage en blocs de 16 Mo sur l'ensemble des sources ingérées
remote_iter = fs.listFiles(raw_hdfs_dir, True)
statuses = []
while remote_iter.hasNext():
    statuses.append(remote_iter.next())
statuses.sort(key=lambda s: s.getLen(), reverse=True)
largest_hdfs_path = statuses[0].getPath().toString()

for status in statuses:
    path = status.getPath()
    rel_hdfs = path.toString().split("/lakehouse/bronze/raw/", 1)[-1]
    length_mb = status.getLen() / (1024 * 1024)
    blocks = fs.getFileBlockLocations(status, 0, status.getLen())
    print(f"Fichier : {rel_hdfs} ({length_mb:.2f} Mo | repl={status.getReplication()})")
    for idx, blk in enumerate(blocks):
        print(f"   Bloc #{idx} ({blk.getLength() / (1024*1024):5.2f} Mo) -> DataNodes : {list(blk.getHosts())}")

Fichier : steam-reviews/steam_reviews.csv (100.35 Mo | repl=2)
   Bloc #0 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #1 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode3']
   Bloc #2 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode2']
   Bloc #3 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode3']
   Bloc #4 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode2']
   Bloc #5 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #6 ( 4.35 Mo) -> DataNodes : ['datanode3', 'datanode2']
Fichier : steam-store-games/steam_games.csv (5.55 Mo | repl=2)
   Bloc #0 ( 5.55 Mo) -> DataNodes : ['datanode1', 'datanode2']


### Questions d'analyse (à vérifier aussi sur http://localhost:9870)
1. Comparez sur l'interface Web du NameNode (`Utilities > Browse the file system`) le nombre de blocs de votre fichier le plus volumineux avec sa taille divisée par 16 Mo.
2. Un fichier de 3 Mo bloque-t-il réellement 16 Mo d'espace disque sur les DataNodes ? Pourquoi ?
3. Pourquoi avoir configuré `dfs.replication = 2` sur un cluster à `3` DataNodes ?


#### Réponses du groupe *(mesures du 2026-11-16 : sortie de la cellule précédente, `hdfs fsck`, `hdfs dfs -du`, UI :9870)*

**1. Nombre de blocs du plus gros fichier vs taille / 16 Mo**

| Fichier | Taille exacte | Taille / 16 Mo | Blocs HDFS observés |
|---|---|---|---|
| `steam-reviews/steam_reviews.csv` | 105 220 762 o (100,35 Mio) | 105 220 762 / 16 777 216 = **6,27** | **7** = 6 blocs pleins de 16 Mo + 1 bloc de 4,35 Mo (4 557 466 o) |
| `steam-store-games/steam_games.csv` | 5 815 053 o (5,55 Mio) | 0,35 | **1** bloc de 5,55 Mo |

Le nombre de blocs vaut donc **⌈taille / dfs.blocksize⌉** : HDFS découpe le flux d'écriture en blocs de taille fixe, et seul le **dernier bloc est partiel**. L'UI du NameNode (*Browse the file system → Block information*) liste bien les blocs 0 à 6 pour `steam_reviews.csv`, chacun sur 2 DataNodes différents (ex. bloc 0 → `datanode3`, `datanode1`). Les 2 fichiers bruts représentent 8 blocs × 2 répliques = 16 répliques réparties sur les 3 nœuds.

**2. Un petit fichier (`steam_games.csv`, 5,55 Mo) occupe-t-il 16 Mo sur disque ? → Non.**

`dfs.blocksize = 16 Mo` est une **taille maximale logique** (l'unité de découpage et de métadonnées gérée par le NameNode), **pas une pré-allocation**. Chaque DataNode stocke un bloc comme un **fichier ordinaire** sur son système de fichiers local (ext4, blocs de 4 Ko) :
- sur `datanode3`, le fichier `blk_1073741832` pèse **5 815 053 o**, exactement la taille du CSV, plus un fichier `blk_1073741832_1008.meta` de 45 439 o (sommes de contrôle CRC : 4 o par tranche de 512 o) ;
- `hdfs dfs -du -h` affiche `5.5 M` (taille logique) et `11.1 M` (espace réellement consommé = 2 répliques), et non 32 Mo.

Le vrai coût des petits fichiers n'est donc **pas l'espace disque** mais la **mémoire du NameNode** (chaque fichier et chaque bloc ≈ 150 o d'objets en RAM, quelle que soit sa taille) et le **parallélisme** (1 fichier ≈ 1 tâche Spark, avec un coût d'ouverture fixe) : c'est le *small files problem* qu'on traitera au Jour 3 avec `OPTIMIZE`.

**3. Pourquoi `dfs.replication = 2` sur 3 DataNodes ?**

C'est un compromis **tolérance aux pannes / coût de stockage / capacité d'auto-réparation** :
- **Tolérance** : chaque bloc existe sur 2 nœuds distincts → la perte de **n'importe quel** DataNode ne fait perdre aucune donnée (démontré au crash-test ci-dessous : 0 bloc manquant). Avec `replication = 1`, la panne de `datanode2` aurait rendu illisibles les blocs qu'il était seul à porter.
- **Coût** : ×2 au lieu de ×3 (défaut Hadoop) → `steam_reviews.csv` consomme 200,7 Mo au lieu de 301 Mo, ce qui compte sur un cluster de TP aux ressources limitées.
- **Auto-réparation observable** : avec `replication = 3` sur 3 nœuds, chaque nœud porterait une copie de tout ; après une panne il ne resterait que 2 nœuds et le facteur 3 serait **impossible à rétablir** (blocs sous-répliqués en permanence). Avec 2 répliques sur 3 nœuds, il reste toujours un **3e nœud cible** : le NameNode peut recopier les blocs orphelins et revenir à l'état nominal (observé en ~13 s après la détection).

En production (dizaines de nœuds, plusieurs racks), on garde `replication = 3` avec *rack awareness* (2 racks distincts) pour survivre à la perte d'un rack entier.

---
## 4. Crash-Test en Direct : Panne d'un DataNode et Auto-Réparation (~50s)

Dans `config/hdfs-site.xml`, le délai de détection d'un DataNode mort a été abaissé de 10m30s à **~50 secondes** :
$$2 \times \text{recheck-interval (10s)} + 10 \times \text{heartbeat.interval (3s)} = 50\text{ s}$$

### Protocole à réaliser depuis votre terminal :
1. Inspectez l'état initial des blocs : `task hdfs:fsck`
2. Arrêtez brutalement le DataNode 2 :
   ```bash
   task hdfs:crash-dn2   # équivalent à : docker stop datanode2
   ```
3. Observez l'onglet **Datanodes** sur `http://localhost:9870` pendant ~50s : dès que `datanode2` passe à **Dead**, le NameNode détecte les blocs sous-répliqués et ordonne leur copie automatique vers le 3e DataNode survivant !

### Exercice 4.1 — Vérifier la continuité de service et la *Data Locality*
Pendant que `datanode2` est éteint, lisez votre plus gros fichier (`largest_hdfs_path`) avec PySpark, affichez `df_check.count()` ainsi que `df_check.rdd.getNumPartitions()`, et expliquez le lien avec le nombre de blocs HDFS.

*(Pensez ensuite à rallumer le nœud avec `task hdfs:restore-dn2` !)*

In [6]:
print("Fichier cible du test de résilience :", largest_hdfs_path)

# CSV brut lu directement sur HDFS. `escape='"'` : le CSV Kaggle échappe les guillemets internes en "".
df_check = spark.read.option("header", "true").option("escape", '"').csv(largest_hdfs_path)

assert df_check is not None, "TODO : Lisez `largest_hdfs_path` dans `df_check`."

print(f"Nombre total de lignes lues       : {df_check.count():,}")
print(f"Nombre de partitions Spark créées : {df_check.rdd.getNumPartitions()}")

# Lien partitions Spark <-> blocs HDFS
largest_status = fs.getFileStatus(PathHDFS(largest_hdfs_path))
n_blocks = len(fs.getFileBlockLocations(largest_status, 0, largest_status.getLen()))
print(f"Nombre de blocs HDFS du fichier   : {n_blocks}")

# Par défaut, taille de split = min(maxPartitionBytes=128 Mo, max(openCost=4 Mo, taille / nb_cœurs)) :
# le nombre de partitions dépend donc des cœurs du driver. En alignant le split sur dfs.blocksize
# (et minPartitionNum=1), on obtient exactement 1 partition = 1 bloc HDFS = 1 tâche.
spark.conf.set("spark.sql.files.maxPartitionBytes", block_size)
spark.conf.set("spark.sql.files.minPartitionNum", "1")
n_parts_aligned = (
    spark.read.option("header", "true").option("escape", '"').csv(largest_hdfs_path).rdd.getNumPartitions()
)
spark.conf.unset("spark.sql.files.maxPartitionBytes")
spark.conf.unset("spark.sql.files.minPartitionNum")
print(f"Partitions avec split = 16 Mo     : {n_parts_aligned}")
assert n_parts_aligned == n_blocks, "Split de 16 Mo : 1 partition Spark doit correspondre à 1 bloc HDFS"


Fichier cible du test de résilience : hdfs://namenode:9000/lakehouse/bronze/raw/steam-reviews/steam_reviews.csv


Nombre total de lignes lues       : 3,763,740
Nombre de partitions Spark créées : 8
Nombre de blocs HDFS du fichier   : 7
Partitions avec split = 16 Mo     : 7


#### Compte-rendu du crash-test `datanode2` *(réalisé le 2026-11-16 depuis le terminal hôte)*

Protocole : `task hdfs:fsck` (état initial : **HEALTHY**, 3 DataNodes, 22 blocs dans `/lakehouse/bronze`, 0 sous-répliqué) → `task hdfs:crash-dn2` → lecture PySpark **pendant la panne** (même code que la cellule ci-dessus + lecture des tables Delta Bronze) → suivi `hdfs dfsadmin -report` / `hdfs fsck` → `task hdfs:restore-dn2`.

| Instant (t0 = `docker stop datanode2`) | Observation |
|---|---|
| t0 → ~55 s | Lectures **réussies** alors que `datanode2` est éteint : `steam_reviews.csv` → **3 763 740 lignes, 8 partitions** ; Delta `bronze/steam_reviews` → 3 763 740 ; Delta `bronze/steam_games` → 27 075. Même `steam_games.csv`, dont l'unique bloc était sur `datanode2` + `datanode3`, est lu (27 075 lignes) via la réplique de `datanode3`. |
| ≤ 55 s | NameNode : **Live datanodes (2) / Dead datanodes (1)** → cohérent avec le délai théorique 2 × 10 s + 10 × 3 s = **50 s** (plus le temps écoulé depuis le dernier *heartbeat*). |
| 63 s | `fsck` : Status **HEALTHY**, **0 bloc manquant**, 3 blocs encore sous-répliqués (13,6 %) : la ré-réplication est déjà en cours. |
| 76 s | **Under-replicated = 0** : les 7 blocs de `steam_reviews.csv` et le bloc de `steam_games.csv` ont de nouveau 2 répliques, toutes sur `datanode1` + `datanode3`. Relecture : mêmes comptes. |
| restore + 10 s | `datanode2` ré-enregistré : **3 DataNodes vivants**, 0 bloc sous/sur-répliqué. Ses anciennes répliques, devenues **excédentaires** (3 copies > facteur 2), ont été invalidées par le NameNode : `datanode2` revient avec 0 bloc, `datanode1` et `datanode3` en portent 22 chacun. |

**Pourquoi la lecture continue pendant la panne ?** Pour chaque bloc, le NameNode renvoie au client la **liste de tous les DataNodes** qui le détiennent. Le client HDFS (`DFSInputStream`) tente le premier ; si la connexion à `datanode2` échoue, il le marque comme mort localement et **bascule sur l'autre réplique**, de façon transparente pour Spark. Cela marche **avant même** que le NameNode ait déclaré le nœud mort (il n'a besoin que de la seconde copie) : la réplication = 2 garantit qu'il en reste toujours une. Une fois `datanode2` déclaré *Dead*, le `ReplicationMonitor` du NameNode ordonne aux nœuds survivants de copier les blocs sous-répliqués vers le 3e DataNode : c'est l'**auto-réparation**.

**Partitions Spark et blocs HDFS (Exercice 4.1).** Le fichier a **7 blocs** mais Spark crée **8 partitions** par défaut : la taille de split vaut `min(maxPartitionBytes = 128 Mo, max(openCost = 4 Mo, 100,35 Mo / 8 cœurs ≈ 12,5 Mo))` ≈ 12,5 Mo, donc 8 splits. En fixant `spark.sql.files.maxPartitionBytes = 16 Mo` (= `dfs.blocksize`) et `minPartitionNum = 1`, on obtient **7 partitions = 7 blocs** (vérifié par l'`assert` ci-dessus) : chaque tâche Spark lit alors exactement un bloc, et l'ordonnanceur peut la placer sur un nœud qui détient ce bloc (*data locality* : on déplace le calcul vers la donnée, pas l'inverse). Le nombre de partitions **ne change pas pendant la panne** (8) : il dépend de la taille du fichier, pas du nombre de répliques disponibles ; seule la source physique de chaque bloc change.

> Dans ce TP, Spark tourne en `local[*]` dans le conteneur `spark-jupyter` (distinct des DataNodes) : la localité reste un principe ici, alors que sur un vrai cluster YARN/Spark les *executors* tourneraient sur les DataNodes eux-mêmes.

La cellule suivante vérifie, après `task hdfs:restore-dn2`, que le cluster est revenu à l'état nominal.


In [7]:
# Contrôle post-crash-test : datanode2 rallumé (task hdfs:restore-dn2) -> 3 DataNodes vivants, 0 bloc perdu.
# Lecture des métriques JMX du NameNode (mêmes chiffres que l'onglet Datanodes de http://localhost:9870).
import json
import time
import urllib.request

JMX = "http://namenode:9870/jmx?qry=Hadoop:service=NameNode,name="


def nn_metric(bean: str, key: str) -> int:
    with urllib.request.urlopen(JMX + bean, timeout=10) as resp:
        return int(json.load(resp)["beans"][0][key])


for _ in range(40):  # jusqu'à ~120 s pour laisser datanode2 se ré-enregistrer
    live = nn_metric("FSNamesystemState", "NumLiveDataNodes")
    if live == 3:
        break
    time.sleep(3)

missing = nn_metric("FSNamesystem", "MissingBlocks")
under_rep = nn_metric("FSNamesystem", "UnderReplicatedBlocks")
print(f"DataNodes vivants : {live} | blocs manquants : {missing} | blocs sous-répliqués : {under_rep}")
assert live == 3, "datanode2 n'est pas revenu : lancez `task hdfs:restore-dn2`"
assert missing == 0, "Des blocs HDFS sont perdus !"


DataNodes vivants : 3 | blocs manquants : 0 | blocs sous-répliqués : 0


---
## 5. Construction de votre Couche Bronze en Delta Lake (`/lakehouse/bronze/`)

### Exercice 5.1 — Convertir vos fichiers bruts (mono ou multi-datasets) en tables Delta Lake Bronze
Dans une architecture **Médaillon multi-sources**, la couche **Bronze** conserve chaque table brute fidèle à sa source tout en l'enrichissant de métadonnées de lignage (*lineage*) indispensables avant les jointures du Jour 2 :
- `_ingested_at` : `F.current_timestamp()`
- `_source_dataset` : `F.lit(dataset_name)` *(identifie le dataset d'origine)*
- `_source_file` : `F.col("_metadata.file_path")`

> **Attention aux noms de colonnes Delta/Parquet :** Si vos fichiers CSV bruts contiennent des espaces ou caractères spéciaux dans les en-têtes, normalisez-les simplement en `snake_case` (ex. `.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])`).

Complétez la cellule ci-dessous pour ingérer chacune des tables utiles de vos datasets dans `/lakehouse/bronze/<nom_table>`.

In [8]:
# Tables Bronze : "nom_table_bronze": ("nom_dataset_source", "/lakehouse/bronze/raw/<sous-dossier>/<fichier>")
BRONZE_SOURCES: dict[str, tuple[str, str]] = {
    "steam_reviews": ("steam_reviews", "/lakehouse/bronze/raw/steam-reviews/steam_reviews.csv"),  # faits
    # dimension
    "steam_games": ("steam_store_games", "/lakehouse/bronze/raw/steam-store-games/steam_games.csv"),
}

assert len(BRONZE_SOURCES) > 0, "TODO : Renseignez vos tables à ingérer dans `BRONZE_SOURCES` (idéalement >= 2 tables à joindre au Jour 2)."

for table_name, source_spec in BRONZE_SOURCES.items():
    dataset_name, raw_path = source_spec if isinstance(source_spec, tuple) else ("default", source_spec)
    bronze_delta_path = f"/lakehouse/bronze/{table_name}"

    # 1. Lecture du CSV brut (en-tête, schéma inféré, guillemets internes doublés "")
    df_raw = (
        spark.read.option("header", "true")
        .option("inferSchema", "true")
        .option("escape", '"')
        .csv(raw_path)
    )

    assert df_raw is not None, f"TODO : Construisez et écrivez la table Bronze `{table_name}`."

    # 2. Noms de colonnes en snake_case + 3. métadonnées de lignage.
    # Un seul select directement sur le scan : la colonne cachée `_metadata` y reste résolvable.
    snake_cols = [re.sub(r"[^0-9a-z]+", "_", c.strip().lower()).strip("_") for c in df_raw.columns]
    df_bronze = df_raw.select(
        *[df_raw[c].alias(s) for c, s in zip(df_raw.columns, snake_cols)],
        F.current_timestamp().alias("_ingested_at"),
        F.lit(dataset_name).alias("_source_dataset"),
        F.col("_metadata.file_path").alias("_source_file"),
    )

    # 4. Écriture Delta idempotente (overwrite) : ré-exécuter le notebook ne duplique pas les données
    (
        df_bronze.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(bronze_delta_path)
    )

    cnt = spark.read.format("delta").load(bronze_delta_path).count()
    print(f"[BRONZE OK] {bronze_delta_path:<38} (source={dataset_name:<18}) -> {cnt:>10,} lignes")


26/11/18 09:46:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[BRONZE OK] /lakehouse/bronze/steam_reviews        (source=steam_reviews     ) ->  3,763,740 lignes


[BRONZE OK] /lakehouse/bronze/steam_games          (source=steam_store_games ) ->     27,075 lignes


In [9]:
# Auto-contrôle de fin de Jour 1 : au moins une table Delta valide dans /lakehouse/bronze/
bronze_root = PathHDFS("/lakehouse/bronze")
bronze_delta_tables = [
    s.getPath().getName()
    for s in fs.listStatus(bronze_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(bronze_delta_tables) > 0, "Aucune table Delta détectée dans /lakehouse/bronze/ !"
print("Tables Delta Bronze validées :", bronze_delta_tables)

Tables Delta Bronze validées : ['steam_games', 'steam_reviews']
